In [ ]:
from st_router.paths import get_rootdir
from st_router.scoring import label_rows, score, run_mcnemar
from st_router.prompt import make_prompt, run_llm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
from dotenv import load_dotenv
import groq
from groq import Groq
from time import sleep
import re
import logging
logging.basicConfig(level=logging.INFO)


ROOT_DIR = get_rootdir()
DATA_RAW = ROOT_DIR / "data" / "raw"
DATA_PROC = ROOT_DIR / "data" / "processed"
DATA_INTERIM = ROOT_DIR / "data" / "interim"

load_dotenv()

In [ ]:
bash = pd.read_parquet(DATA_PROC / "bash_windows.parquet")

df = bash.copy()
df = df[df["window_len"] != 0]

# randomly select one row per user - 267 users
df = df.groupby("unique_key").sample(random_state=42)
df.head()

In [ ]:
DATA_INTERIM.mkdir(parents=True, exist_ok=True)
prompt = make_prompt()

out20 = DATA_INTERIM / "base_rate_gptoss20b.jsonl"
if not out20.exists():
    preds_oss20b = run_llm(df, prompt)
    oss20b = pd.DataFrame(preds_oss20b)
    oss20b.to_json(out20, orient="records", lines=True)

out120 = DATA_INTERIM / "base_rate_gptoss120b.jsonl"
if not out120.exists():
    preds_oss120b = run_llm(df, prompt, model="openai/gpt-oss-120b")
    oss120b = pd.DataFrame(preds_oss120b)
    oss120b.to_json(out120, orient="records", lines=True)

out120_medium = DATA_INTERIM / "base_rate_gptoss120_medium.jsonl"
if not out120_medium.exists():
    preds_oss120b_medium = run_llm(df, prompt, reasoning_effort="medium", model="openai/gpt-oss-120b", sleep_s=10)
    oss120b_medium = pd.DataFrame(preds_oss120b_medium)
    oss120b_medium.to_json(out120_medium, orient="records", lines=True)

# Exploratory

In [ ]:
df = pd.read_json(DATA_INTERIM / "base_rate_gptoss20b.jsonl", lines=True)
df["prediction"] = df["prediction"].str.strip()
df["actual"] = df["actual"].str.strip()
accuracy = (df["actual"] == df["prediction"]).sum()
print(f"exact prediction accuracy {accuracy/df.shape[0]*100}%")

Exact matching is very harsh with only 9% accuracy rate on the test set. We need to investigate what it gets wrong

In [ ]:
print(df["prediction"].isna().sum())
print(df["actual"].isna().sum())
# one space
print((df["prediction"] == " ").sum())
print((df["actual"] == " ").sum())
# empty string      
print((df["prediction"] == "").sum())
print((df["actual"] == "").sum())

display(df["prediction"].value_counts())
display(df["actual"].value_counts())

Exact matches not a good metric. a lot of the commands entered in actual aren't predictable e.g passwords, typos. We rely on the work of Davison & Hirsh (1998), "Predicting Sequences of User Actions", Korvemaker & Greiner (2000), "Predicting UNIX Command Lines: Adjusting to User Patterns", Lin et al. (2018), NL2Bash.

We need to count how many of the misses are: 1) same command with different flags/arguments 2) different command

In [ ]:
cmd_pred = df["prediction"].str.replace(r"^sudo\s+", "", regex=True).str.split().str[0]
cmd_actual = df["actual"].str.replace(r"^sudo\s+", "", regex=True).str.split().str[0]

print(f"command matches: {(cmd_pred == cmd_actual).mean() * 100}%")

In [ ]:
match = cmd_pred == cmd_actual
display(cmd_actual[match].value_counts())

We decide to score:
+ Exact: compare full string
+ Command-name: strip whitespace, strip sudo, compare first token (command)

Three target cateogies: exact, edited, different

## TODO
As we scale a lookup table will classify if a command is valid and check if it is a typo (within 1 Damerau–Levenshtein distance), or invalid command e.g. password. Typos in the actual values will be then scored against the actual predicted commands by using the lookup table for the precise syntax.

This will be done by eye in our small test set

In [ ]:
with pd.option_context("display.max_rows", None):
    display(cmd_actual[~match].value_counts())

# Scoring

2 clean typos (mfsconsole, ipconfig) of 10 problem rows

Main limitation: smart-terminal has more context than the model. We only give it the working direcotyr while smart terminal has:

+ cwd_contents: the files and folders in the current directory. This is the big one.
+ shell_tools (tool versions), plus shell and os.
+ Memory of past suggestions in that project.
+ Tools, in particular ReadLastError, which reads the last error message.

# Cheap Model: GPT-OSS-20b

In [ ]:
df = pd.read_json(DATA_INTERIM / "base_rate_gptoss20b.jsonl", lines=True)
df = label_rows(df)
scored = df[df["label"] != "non_command"].copy()
exact, cmd_match = score(scored["prediction"], scored["target"])

print(f"exact string matches: {exact.mean() * 100:.1f}%")
print(f"command matches: {cmd_match.mean() * 100:.1f}%")

scored["prev_1"] = bash.loc[scored["idx"], "prev_1"].values
exact_base, cmd_base = score(scored["prev_1"], scored["target"])

print(f"exact string matches: {exact_base.mean() * 100:.1f}%")
print(f"command matches: {cmd_base.mean() * 100:.1f}%")

model_exact, model_cmd = score(scored["prediction"], scored["target"])
base_exact, base_cmd = score(scored["prev_1"], scored["target"])

run_mcnemar(model_exact, base_exact, "exact")
run_mcnemar(model_cmd, base_cmd, "command")

# Expensive Model: GPT-OSS-120b

In [ ]:
df = pd.read_json(DATA_INTERIM / "base_rate_gptoss120b.jsonl", lines=True)
df = label_rows(df)

scored = df[df["label"] != "non_command"].copy()
exact, cmd_match = score(scored["prediction"], scored["target"])

print(f"exact string matches: {exact.mean() * 100:.1f}%")
print(f"command matches: {cmd_match.mean() * 100:.1f}%")

scored["prev_1"] = bash.loc[scored["idx"], "prev_1"].values
exact_base, cmd_base = score(scored["prev_1"], scored["target"])

print(f"exact string matches: {exact_base.mean() * 100:.1f}%")
print(f"command matches: {cmd_base.mean() * 100:.1f}%")


model_exact, model_cmd = score(scored["prediction"], scored["target"])
base_exact, base_cmd = score(scored["prev_1"], scored["target"])

run_mcnemar(model_exact, base_exact, "exact")
run_mcnemar(model_cmd, base_cmd, "command")